# 03 Broyden 拟 Newton 与参数延拓

Newton 法每步重新计算 Jacobian，弦 Newton 则完全固定 Jacobian。Broyden 方法位于两者之间：从一个 Jacobian 近似开始，用迭代过程中得到的割线信息更新它。参数延拓则利用另一个常见事实：如果问题依赖参数，邻近参数的解通常也相近，可以把上一参数的解作为下一次求解的初值。


In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
while ROOT.name != "py-sc" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import broyden_system_method, newton_system_method, parameter_continuation


## Broyden 拟 Newton 方法

设当前 Jacobian 近似为 $B_k$。求解

$$
B_k s_k=-F(x_k),\qquad x_{k+1}=x_k+s_k.
$$

得到

$$
y_k=F(x_{k+1})-F(x_k).
$$

新的近似矩阵应尽量满足割线条件

$$
B_{k+1}s_k=y_k.
$$

一个常见的“好 Broyden”更新为

$$
B_{k+1}=B_k+\frac{(y_k-B_ks_k)s_k^T}{s_k^Ts_k}.
$$


In [ ]:
def F(x):
    return np.array([x[0] ** 2 + x[1] ** 2 - 1.0, x[0] - x[1]])


def J(x):
    return np.array([[2.0 * x[0], 2.0 * x[1]], [1.0, -1.0]])

broyden = broyden_system_method(F, [0.8, 0.6], tolerance=1e-10)
newton = newton_system_method(F, J, [0.8, 0.6], tolerance=1e-12)
expected = np.array([1.0 / math.sqrt(2.0), 1.0 / math.sqrt(2.0)])

print("Broyden solution:", np.array2string(broyden.solution, precision=12))
print("expected:", np.array2string(expected, precision=12))
print("Broyden iterations:", broyden.iterations, "residual:", f"{broyden.residual_norm:.3e}")
print("Newton iterations:", newton.iterations)
print("Broyden residual history:", np.array2string(broyden.residual_history, precision=3))


## 参数延拓

考虑一族方程组

$$
F(\lambda, x)=0.
$$

若参数从 $\lambda_i$ 逐步变化到 $\lambda_{i+1}$，且解分支连续，则可用 $x(\lambda_i)$ 作为求解 $x(\lambda_{i+1})$ 的初值。这种 predictor 为“零阶预测”，实际延拓方法还会加入切向预测和步长控制。


In [ ]:
def parameter_system(parameter, x):
    return np.array([x[0] ** 2 - parameter, x[1] - parameter])

parameters = np.array([1.0, 1.5, 2.0, 2.5])
continuation = parameter_continuation(parameter_system, parameters, initial=[1.0, 1.0], tolerance=1e-10)
expected_solutions = np.column_stack((np.sqrt(parameters), parameters))

print("parameters:", continuation.parameters)
print("solutions:")
print(np.array2string(continuation.solutions, precision=10))
print("expected:")
print(np.array2string(expected_solutions, precision=10))
print("max residual:", f"{np.max(continuation.residual_norms):.3e}")


## 适用范围

Broyden 方法减少了 Jacobian 计算或分解次数，但近似矩阵可能退化，需要重启、阻尼或预条件策略。参数延拓适合解分支平滑的问题；若路径上出现转折、分岔或奇异 Jacobian，仅用上一解作为下一初值可能失败，需要更完整的伪弧长延拓。


In [ ]:
print("Broyden converged:", broyden.converged)
print("Continuation converged:", continuation.converged)


## 本节小结

Broyden 方法用割线条件更新 Jacobian 近似，在每步成本和收敛速度之间折中。参数延拓利用相邻参数解的连续性，把一串难问题拆成逐步变化的小问题。二者都体现了非线性方程组计算中的实用思想：不要只看单次迭代公式，还要利用已经得到的矩阵、方向和路径信息。
